# 24 · Sweep окон 1..23ч с ИДЕНТИЧНЫМ §9 построением таргета

В ноутбуке 22 таргет строился «эпизодно» (серьёзность раздавалась всем каналам
объекта) — некорректно сравнимо с §9. Здесь — тот же принцип, что в §9:

**событие у канала в бакете t = 1**, если
1. у канала в этом 6ч-бакете есть «Обнаружен дым» (или «Рычаг сдернут») И
2. на объекте в окне `[t·6ч, t·6ч + h·3600)` задымились ≥2 РАЗНЫХ дымовых
   канала (включая свой).

При h=6 это в точности колонка `серьёзных` из §9 — проверим совпадением базы и
метрик. Затем прогоняем h=1..23 одним протоколом (GPU 900/0.03/10, ES=250,
sample 12000/8000/8000) и ищем оптимум окна в РАМКАХ этой методики.


In [1]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import survival_metrics, pr_at_horizon


DATASET = du.find_dataset_dir()
hour_ns = fe.BUCKET_HOURS * 3600 * 10 ** 9
EV = pd.read_parquet(DATASET / "fire_raw_events_30m.parquet")
panel_base = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                         dtype={"ид_канала_данных": str, "ид_объект": str})
panel_base = tte.refit_z_train_only(panel_base)
print("сырых событий:", len(EV), "| панель:", panel_base.shape)


сырых событий: 164477 | панель: (1204508, 143)


In [2]:
def serious_map(h_sec):
    # (об, бакет, канал) серьёзных подтверждённых событий + ручные (как в §9)
    ev = EV[EV["тип"] == "дым"].copy()
    ev["бакет"] = (ev["t"] // hour_ns).astype(np.int64)
    rows = []
    for об, g in ev.groupby("об"):
        g = g.sort_values("t")
        t = g["t"].to_numpy()
        ids = g["ид"].to_numpy()
        bk = g["бакет"].to_numpy()
        left = bk * hour_ns
        right = left + h_sec
        lo = np.searchsorted(t, left, side="left")
        hi = np.searchsorted(t, right, side="right")
        for i in range(len(g)):
            if hi[i] - lo[i] < 2:
                continue
            chans = np.unique(ids[lo[i]:hi[i]])
            if len(chans) >= 2:
                for c in chans:
                    rows.append((об, bk[i], c))
    conf = pd.DataFrame(rows, columns=["об", "бакет", "канал"]).drop_duplicates()
    # ручные — событие всегда серьёзное у своего канала
    man = EV[EV["тип"] == "ручной"].copy()
    if len(man):
        man["бакет"] = (man["t"] // hour_ns).astype(np.int64)
        man = man[["об", "бакет", "ид"]].rename(columns={"ид": "канал"}).drop_duplicates()
        conf = pd.concat([conf, man], ignore_index=True).drop_duplicates()
    conf["об"] = conf["об"].astype(str)
    conf["канал"] = conf["канал"].astype(str)
    return conf


In [3]:
def make_col(panel, h_sec):
    cm = serious_map(h_sec)
    cm = cm.rename(columns={"об": "ид_объект", "канал": "ид_канала_данных"})
    pn = panel.copy()
    pn = pn.merge(cm, on=["ид_объект", "бакет", "ид_канала_данных"],
                  how="left", indicator="m")
    pn["серьёзных_h"] = (pn["m"] == "both").astype(np.int8)
    pn = pn.drop(columns=["m"])
    return pn

# ВАЛИДАЦИЯ: h=6 должен совпасть с колонкой §9 (база 8.522%)
pn6 = make_col(panel_base, 6 * 3600 * 10 ** 9)
print("база h=6 (наш метод): {:.4f}% | §9-колонка: {:.4f}%".format(
    (pn6["серьёзных_h"] > 0).mean() * 100, panel_base["серьёзных"].mean() * 100))
print("совпадение с §9 (доля строк): {:.4f}%".format(
    (pn6["серьёзных_h"].to_numpy() == panel_base["серьёзных"].to_numpy()).mean() * 100))


база h=6 (наш метод): 8.5219% | §9-колонка: 8.5217%
совпадение с §9 (доля строк): 99.9998%


In [4]:
def run(pn, col="серьёзных_h"):
    p = tte.add_series_context(pn.copy(), event_col=col)
    s = tte.add_tte_labels(p)
    s["ид_объект_code"] = s["ид_объект"].astype("category").cat.codes
    s["тип_датчика_code"] = s["тип_датчика"].astype("category").cat.codes
    tr, va, ho = tte.split_subjects(s)
    Xc = tte.subject_features(tr)
    trs = tte.sample_subjects(tr, 12000, 8000, 8000, seed=42)
    vas = tte.sample_subjects(va, 2000, 2000, 2000, seed=43)
    hos = tte.sample_holdout(ho, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(trs, Xc, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(vas, Xc, tte.HORIZON_BUCKETS)
    m = CatBoostClassifier(iterations=900, learning_rate=0.03, depth=10,
                           random_seed=42, task_type="GPU",
                           loss_function="Logloss", eval_metric="Logloss",
                           early_stopping_rounds=250, verbose=0,
                           allow_writing_files=False)
    m.fit(Pool(X_tr, y_tr), eval_set=Pool(X_va, y_va))
    surviv_tr = tte.make_surv_struct(trs["event_flag"], trs["obs_days"])
    surviv_ho = tte.make_surv_struct(hos["event_flag"], hos["obs_days"])
    Xh, _ = tte.expand_person_time(hos, Xc, tte.HORIZON_BUCKETS, fixed_horizon=True)
    S = np.cumprod(1 - m.predict_proba(Xh)[:, 1].reshape(-1, tte.HORIZON_BUCKETS), axis=1)
    met = survival_metrics(surviv_tr, surviv_ho, 1 - S[:, -1], S, tte.EVAL_TIMES)
    y24 = ((hos["event_flag"] == 1) & (hos["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    return met, pr, hos, y24, p24

rows = []
for h in range(1, 24):
    pn = make_col(panel_base, h * 3600 * 10 ** 9)
    met, pr, hos, y24, p24 = run(pn)
    base_tr = ((pn["серьёзных_h"].to_numpy() > 0))  # bc не нужно
    rows.append({"h": h, "база_holdout24%": round(pr["base"] * 100, 3),
                 "uno_c": round(met["uno_c"], 4),
                 "pr_auc24": round(pr["pr_auc"], 4)})
    print(f"h={h:2d}: PR24={pr['pr_auc']:.3f} Uno-C={met['uno_c']:.3f} "
          f"база24_ho={pr['base']*100:.2f}%", flush=True)

RES = pd.DataFrame(rows)
RES.to_csv(DATASET / "fire_window_sweep_s9.csv", index=False, encoding="utf-8-sig")
print()
print(RES.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
best = RES.loc[RES["pr_auc24"].idxmax()]
print("лучшее окно по PR24:", int(best["h"]), "PR24=", round(best["pr_auc24"], 3))


h= 1: PR24=0.075 Uno-C=0.765 база24_ho=0.39%


h= 2: PR24=0.255 Uno-C=0.826 база24_ho=0.57%


h= 3: PR24=0.373 Uno-C=0.817 база24_ho=0.67%


h= 4: PR24=0.245 Uno-C=0.794 база24_ho=0.85%


h= 5: PR24=0.281 Uno-C=0.817 база24_ho=1.30%


...

h=20: PR24=0.306 Uno-C=0.813 база24_ho=1.01%


h=21: PR24=0.102 Uno-C=0.845 база24_ho=1.01%


h=22: PR24=0.173 Uno-C=0.809 база24_ho=1.00%


h=23: PR24=0.098 Uno-C=0.778 база24_ho=0.92%



 h  база_holdout24%  uno_c  pr_auc24
 1           0.3850 0.7653    0.0754
 2           0.5650 0.8257    0.2552
 3           0.6700 0.8167    0.3733
 4           0.8500 0.7937    0.2453
 5           1.3000 0.8174    0.2807
 6           1.5300 0.8263    0.3607
 7           1.4600 0.8405    0.3959
 8           1.3400 0.8175    0.0768
 9           1.3000 0.8252    0.1949
10           1.2650 0.7966    0.1275
11           1.1400 0.7974    0.3536
12           1.0900 0.7927    0.2058
13           1.0900 0.8184    0.2264
14           1.0850 0.8100    0.1423
15           1.0800 0.8024    0.1889
16           1.0700 0.8027    0.1599
17           1.0300 0.8086    0.2555
18           1.0050 0.8111    0.2758
19           1.0050 0.8112    0.1233
20           1.0100 0.8129    0.3058
21           1.0100 0.8447    0.1019
22           0.9950 0.8086    0.1734
23           0.9200 0.7776    0.0982
лучшее окно по PR24: 7 PR24= 0.396


### Вывод
- Валидационный шаг отдельно печатает совпадение h=6 с §9-колонкой.
- Если sweep §9-идентичный имеет плато/максимум — выбираем окно соответствующее
  качеству при `pr_auc24` и база24: сравниваем с эталоном §9 (0.421–0.443).
